# Episode 16: Choosing the Right Pattern

You now know every orchestration pattern in AG2 Beta — a single agent, task delegation, and four Hub adapters. This episode is the map: **given a problem, which one do you reach for?**

This is the close of the architecture half of the workshop. From Episode 17 on, you'll *apply* these to real systems.

## Three tiers

Every pattern falls into one of three tiers. Go up a tier only when the one below genuinely can't do the job — each tier adds capability *and* moving parts.

| Tier | Pattern | Who's in control |
|---|---|---|
| **1 — Single agent** | `Agent` + tools | The agent itself |
| **2 — Delegation** | `agent.as_tool()` / `TaskConfig` | One coordinator drives |
| **3 — Networking** | Hub + a channel adapter | Autonomous, registered peers |

Most problems are solved at tier 1 or 2. Tier 3 is for when agents are genuinely independent participants — and it buys you the registry, the audit log, and replayable WALs.

## The decision table

| When you need… | Use |
|---|---|
| One agent with capabilities | `Agent` + `@tool` (Ep 3) |
| Real-time voice | `LiveAgent` (Ep 5) |
| Typed output instead of text | `response_schema` (Ep 9) |
| One agent to orchestrate others | `agent.as_tool()` or `TaskConfig` (Ep 4) |
| Parallel independent sub-tasks | `TaskConfig` + `run_subtasks(parallel=True)` (Ep 4) |
| A strict one-question / one-answer exchange | Hub + **consulting** (Ep 11) |
| Free-form dialogue between two agents | Hub + **conversation** (Ep 12) |
| A fixed-order N-party debate | Hub + **discussion** (Ep 13) |
| A declarative, conditional flow | Hub + **workflow** + `TransitionGraph` (Ep 14) |
| An agent to route requests dynamically | Hub + workflow + **coordinator** (Ep 15) |

## Tier 1 — when one agent is enough

Reach for a single `Agent` when the work is *one cohesive job*, even if it needs tools or multiple steps. Adding agents has a cost — more LLM calls, more prompts to tune, more failure modes.

A single agent is the right answer far more often than it looks. Try it first; split only when you can name *why* one agent is failing (it's juggling conflicting roles, or a step needs genuinely different expertise).

## Tier 2 — when one agent should drive others

Move to **delegation** when there's a clear *owner* of the workflow that wants to hand off focused pieces:

- **`agent.as_tool()`** — when the delegates are *distinct, named* roles the coordinator reasons about ("call the researcher", "call the writer").
- **`TaskConfig`** — when the coordinator should break work into its *own* dynamic sub-tasks, or fan out parallel ones.

The defining trait of tier 2: one agent is unambiguously in charge. The others are tools it calls.

## Tier 3 — when agents are peers

Move to the **Hub** when agents are autonomous participants, not tools — they have their own identity, they collaborate rather than serve, and you want every interaction registered and replayable.

Then pick the adapter by the *shape of the interaction*:

| Adapter | Shape | Closes |
|---|---|---|
| **consulting** | 2 agents, one Q → one A | Auto, after the reply |
| **conversation** | 2 agents, free-form | You close it |
| **discussion** | N agents, fixed round-robin | You close it |
| **workflow** | N agents, a declarative graph | The graph terminates |

And the coordinator pattern (Ep 15) is just a `workflow` whose graph routes through one decision-making agent — use it when *routing itself* needs an LLM.

## A worked example

> *"Build a support system: a request comes in, gets classified, and is handled by the right specialist — billing, technical, or general."*

Walk the tiers:

- **Tier 1?** One agent could try, but billing/technical/general need different knowledge and tone. One prompt juggling all three gets muddy. No.
- **Tier 2?** A coordinator with `agent.as_tool()` for each specialist works well — the coordinator classifies and delegates. **Viable.**
- **Tier 3?** If specialists are independently owned, need an audit trail, or the routing is complex, a `workflow` channel with the coordinator pattern fits. **Also viable — and it's exactly what Episode 17 builds.**

There's rarely one "correct" answer — there's the simplest one that meets the real requirements (audit? independence? scale?). Name the requirement, then pick the lowest tier that satisfies it.

## Don't forget cost

Every tier up multiplies LLM calls. A single agent is one call per turn; a five-agent workflow can be five or more. That's usually fine — the structure is worth it — but it's a real axis:

- Prototyping or latency-critical? Stay low.
- Need auditability, independence, or genuine parallelism? Pay for the higher tier; it earns its keep.

The middleware and assembly tools from Group 2 (token limits, history trimming) apply at *every* tier to keep that cost in check.

## What's next

That's the whole architecture toolkit. **Group 5 — Applications** puts it to work: Episode 17 builds the customer-service system sketched above, end to end.